<a href="https://colab.research.google.com/github/lassyamkotian2006/Ai-College-Academic-Assistant/blob/main/Person2_RAG_SearchEngine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎓 Ai-College-Academic-Assistant — Person 2: RAG & Search Engine Lead

This Google Colab notebook implements the **complete Retrieval & Search Engine subsystem**:

| Step | Component | Description |
|---|---|---|
| 1 | **Dependencies** | Modern LangChain + Google GenAI + FAISS |
| 2 | **API Key** | Gemini API key setup with validation check |
| 3 | **Vector Store** | Load FAISS index (or auto-build demo index) |
| 4 | **LangChain Retriever** | Threshold & MMR retriever setup |
| 5 | **Top-K Search** | Similarity search with relevance scores |
| 6 | **Score Filtering** | Drop weak/irrelevant chunks |
| 7 | **Context Extraction** | Structured citations and formatted context |
| 8 | **RAG Prompt** | College academic persona and grounding |
| 9 | **LCEL RAG Chain** | Full end-to-end LangChain pipeline |
| 10 | **Ask Pipeline** | Complete QA function with source citations |
| 11 | **Test Suite** | 8 standard evaluation queries with pass/fail report |
| 12 | **Interactive Chat** | Real-time student question & answer loop |

## ⚙️ Step 1: Install Dependencies

In [9]:
!pip install -q langchain langchain-core langchain-google-genai langchain-community langchain-text-splitters faiss-cpu google-generativeai pypdf

print('✅ Dependencies installed successfully!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.5/263.5 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth

## 🔑 Step 2: Set Up Google Gemini API Key

Get your free API key from: [https://aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey)

In [10]:
import os
from google.colab import userdata

# 1. Try loading from Colab Secrets (Recommended)
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    print('✅ API key loaded from Colab Secrets!')
except Exception:
    # 2. Or paste your key directly here if needed
    GOOGLE_API_KEY = 'YOUR_GOOGLE_API_KEY_HERE'
    if GOOGLE_API_KEY == 'YOUR_GOOGLE_API_KEY_HERE':
        print('⚠️ Please set your GOOGLE_API_KEY in Colab Secrets or paste it in this cell.')
    else:
        print('✅ Using provided API key!')

os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY

# Validation check
if GOOGLE_API_KEY and GOOGLE_API_KEY != 'YOUR_GOOGLE_API_KEY_HERE':
    try:
        import google.generativeai as genai
        genai.configure(api_key=GOOGLE_API_KEY)
        models = [m.name for m in genai.list_models() if 'embed' in m.name or 'gemini' in m.name]
        print(f'✅ API key verified with Google AI! (Found {len(models)} accessible models)')
    except Exception as e:
        print(f'❌ API key validation error: {e}')

⚠️ Please set your GOOGLE_API_KEY in Colab Secrets or paste it in this cell.


## 📂 Step 3: Load Vector Store (or Auto-Build Demo Index)

If Person 1 shared the `faiss_college_index` folder, we load it. Otherwise, we automatically initialize a sample college handbook vector store so you can test immediately.

In [11]:
import os
from typing import List, Tuple
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS

# ── Config Constants ─────────────────────────────────────────────────────────
EMBEDDING_MODEL   = 'models/embedding-001'   # Google Gemini embedding
QUERY_TASK_TYPE   = 'retrieval_query'         # for querying embeddings
LLM_MODEL         = 'gemini-1.5-flash'
DEFAULT_K         = 5
DEFAULT_THRESHOLD = 0.40
FAISS_INDEX_PATH  = '/content/faiss_college_index'

# Initialize Embedding model
embeddings = GoogleGenerativeAIEmbeddings(
    model=EMBEDDING_MODEL,
    google_api_key=GOOGLE_API_KEY,
    task_type=QUERY_TASK_TYPE,
)

# Check if FAISS index exists on disk, else create a rich demo dataset
if os.path.exists(FAISS_INDEX_PATH) and os.path.exists(os.path.join(FAISS_INDEX_PATH, 'index.faiss')):
    print(f'📂 Found existing FAISS index at: {FAISS_INDEX_PATH}')
    vectorstore = FAISS.load_local(
        FAISS_INDEX_PATH,
        embeddings,
        allow_dangerous_deserialization=True
    )
    print('✅ FAISS index loaded successfully from disk!')
else:
    print('ℹ️ No pre-built FAISS index found. Initializing demo college handbook...')

    DEMO_COLLEGE_DATA = """
    University Academic Handbook 2024-25

    EXAMINATION POLICY:
    All students must maintain a minimum attendance of 75% to be eligible for
    end-semester examinations. Students with attendance below 75% will be detained.
    Internal assessments contribute 40% and end-semester exams contribute 60%
    to the final grade.

    GRADING SYSTEM:
    O  (Outstanding)  : 90-100 marks  → 10 Grade Points
    A+ (Excellent)    : 80-89  marks  → 9  Grade Points
    A  (Very Good)    : 70-79  marks  → 8  Grade Points
    B+ (Good)         : 60-69  marks  → 7  Grade Points
    B  (Above Average): 55-59  marks  → 6  Grade Points
    C  (Average)      : 50-54  marks  → 5  Grade Points
    P  (Pass)         : 45-49  marks  → 4  Grade Points
    F  (Fail)         : Below 45      → 0  Grade Points

    SCHOLARSHIP INFORMATION:
    Merit Scholarship: Students with CGPA >= 9.0 are eligible for a full tuition
    fee waiver. Students with CGPA between 8.0 and 8.9 receive a 50% waiver.
    Applications must be submitted by October 31st each academic year.

    LIBRARY RULES:
    Students can borrow up to 3 books at a time for a 14-day period.
    A fine of Rs. 5 per day is charged for overdue books.
    Reference books and periodicals cannot be borrowed.

    HOSTEL FACILITIES:
    The college provides separate hostels for boys and girls.
    Mess timings: Breakfast 7:00-8:30 AM, Lunch 12:00-1:30 PM, Dinner 7:00-8:30 PM.
    Gate-closing time is 9:00 PM on weekdays and 10:00 PM on weekends.

    COMPUTER SCIENCE DEPARTMENT:
    B.Tech CSE Program Duration: 4 years (8 semesters)
    Total Credits Required: 160
    Core subjects include: Data Structures, Algorithms, DBMS, Operating Systems,
    Computer Networks, Machine Learning, and Software Engineering.
    Electives are offered from Semester 5 onwards.
    """

    splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=120)
    doc = Document(page_content=DEMO_COLLEGE_DATA, metadata={'source': 'Academic_Handbook_2024.pdf', 'page': 1, 'file_type': 'pdf'})
    chunks = splitter.split_documents([doc])
    for i, chunk in enumerate(chunks):
        chunk.metadata['chunk_index'] = i

    print(f'⏳ Embedding {len(chunks)} chunks with Gemini embedding model...')
    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local(FAISS_INDEX_PATH)
    print(f'✅ FAISS index created and saved to {FAISS_INDEX_PATH}')

/tmp/ipykernel_892/3338186034.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


ℹ️ No pre-built FAISS index found. Initializing demo college handbook...
⏳ Embedding 4 chunks with Gemini embedding model...


GoogleGenerativeAIError: Error embedding content (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}

## 🔍 Step 4: LangChain Retriever

Configure a retriever with configurable top-K and score threshold.

In [12]:
def build_retriever(store: FAISS, k: int = DEFAULT_K, threshold: float = DEFAULT_THRESHOLD):
    """Build a threshold-filtered retriever."""
    return store.as_retriever(
        search_type='similarity_score_threshold',
        search_kwargs={'k': k, 'score_threshold': threshold}
    )

retriever = build_retriever(vectorstore, k=DEFAULT_K, threshold=DEFAULT_THRESHOLD)
print(f'✅ LangChain Retriever ready (k={DEFAULT_K}, threshold={DEFAULT_THRESHOLD})')

# Quick sanity test
sample_test = retriever.invoke('What is the minimum attendance requirement?')
print(f'📄 Retrieved {len(sample_test)} relevant chunk(s) for test query.')

NameError: name 'vectorstore' is not defined

## 📊 Step 5: Top-K Similarity Search with Scores

In [ ]:
def top_k_search(store: FAISS, query: str, k: int = DEFAULT_K) -> List[Tuple[Document, float]]:
    """
    Search the FAISS vector database and return (Document, relevance_score) pairs.
    """
    results = store.similarity_search_with_relevance_scores(query, k=k)

    print(f"\n{'='*70}")
    print(f"  🔎 QUERY: {query}")
    print(f"  📋 TOP-{k} RETRIEVED CHUNKS:")
    print(f"{'='*70}")
    for i, (doc, score) in enumerate(results, 1):
        src   = doc.metadata.get('source', 'unknown')
        page  = doc.metadata.get('page', '-')
        ftype = doc.metadata.get('file_type', '?')
        match_status = '✅ PASS' if score >= DEFAULT_THRESHOLD else '❌ LOW SCORE'

        print(f"\n  [{i}] Score: {score:.4f} ({match_status})")
        print(f"      Source: {src} | Page: {page} | Type: {ftype}")
        print(f"      Text: {doc.page_content[:140].strip()}...")
    print(f"{'='*70}\n")
    return results

# Test query
test_results = top_k_search(vectorstore, 'What is the grading system and passing marks?')


## 🎯 Step 6: Similarity Score / Threshold Filtering

In [13]:
def filter_by_score(results: List[Tuple[Document, float]], threshold: float = DEFAULT_THRESHOLD) -> List[Tuple[Document, float]]:
    """
    Filter out any chunks that fall below the relevance threshold to prevent hallucinations.
    """
    filtered = [(doc, score) for doc, score in results if score >= threshold]
    dropped = len(results) - len(filtered)
    print(f"[FILTER] Retained {len(filtered)}/{len(results)} chunks (dropped {dropped} below threshold={threshold})")
    return filtered

# Example demonstration
filtered_chunks = filter_by_score(test_results, threshold=0.40)

NameError: name 'test_results' is not defined

## 📝 Step 7: Context Extraction & Source Citation Formatting

In [14]:
def extract_context(filtered_results: List[Tuple[Document, float]]) -> str:
    """
    Format retrieved chunks with clear metadata citations for injection into the prompt.
    """
    if not filtered_results:
        return 'No relevant context found in the college documents.'

    parts = []
    for doc, score in filtered_results:
        src   = doc.metadata.get('source', 'unknown')
        page  = doc.metadata.get('page', '-')
        ftype = doc.metadata.get('file_type', '?')
        header = f"[Source: {src} | Page: {page} | Confidence: {score:.1%}]"
        parts.append(f"{header}\n{doc.page_content.strip()}")

    return '\n\n───\n\n'.join(parts)

context_str = extract_context(filtered_chunks)
print('Formatted Context Preview:')
print(context_str[:400] + '...')

NameError: name 'filtered_chunks' is not defined

## 💬 Step 8: RAG Prompt Template

In [ ]:
from langchain_core.prompts import PromptTemplate

RAG_PROMPT_TEMPLATE = """\
You are an intelligent College Academic Assistant.
Answer the student's question accurately and concisely based ONLY on the provided college document context.

Guidelines:
- Base your answer strictly on the provided context.
- If the answer cannot be found in the context, reply:
  "I couldn't find that information in the college documents. Please contact the academic office."
- Cite the relevant document source or page where applicable.
- Be friendly, structured, and helpful to students.

Context from College Documents:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{context}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Student Question: {question}

Answer:"""

rag_prompt = PromptTemplate(
    input_variables=['context', 'question'],
    template=RAG_PROMPT_TEMPLATE,
)
print('✅ RAG Prompt template initialized.')

## 🚀 Step 9: Build the End-to-End LCEL RAG Chain

In [46]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Initialize Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.2,
    max_retries=5
)

def format_docs_for_chain(docs: List[Document]) -> str:
    if not docs:
        return 'No relevant context found in the college documents.'
    parts = []
    for doc in docs:
        src  = doc.metadata.get('source', 'unknown')
        page = doc.metadata.get('page', '-')
        parts.append(f"[Source: {src} | Page: {page}]\n{doc.page_content.strip()}")
    return '\n\n───\n\n'.join(parts)

# LangChain LCEL pipeline
rag_chain = (
    {'context': retriever | format_docs_for_chain, 'question': RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print(f'✅ End-to-end RAG Chain built with {LLM_MODEL}!')

NameError: name 'retriever' is not defined

## 🧪 Step 10: Complete ask() QA Function with Citations

In [45]:
def ask(question: str, k: int = DEFAULT_K, threshold: float = DEFAULT_THRESHOLD, show_sources: bool = True) -> str:
    """
    Full pipeline helper: Top-K Search -> Score Filtering -> Context Extraction -> Prompt -> Gemini LLM
    """
    raw_results = top_k_search(vectorstore, question, k=k)
    filtered    = filter_by_score(raw_results, threshold=threshold)
    context     = extract_context(filtered)

    # Generate answer
    prompt_val  = rag_prompt.format(context=context, question=question)
    response    = llm.invoke(prompt_val)
    answer      = response.content if hasattr(response, 'content') else str(response)

    print(f"\n{'='*70}")
    print(f"❓ QUESTION: {question}")
    print(f"{'='*70}")
    print(f"🎓 ASSISTANT ANSWER:\n{answer}")

    if show_sources and filtered:
        print(f"\n📚 SOURCES RETRIEVED ({len(filtered)} chunk(s)):")
        for i, (doc, score) in enumerate(filtered, 1):
            src = doc.metadata.get('source', 'unknown')
            page = doc.metadata.get('page', '-')
            print(f"   [{i}] {src} (p. {page}) — Relevance: {score:.1%}")
    print(f"{'='*70}\n")
    return answer

# Sample execution
ans1 = ask('What is the minimum attendance requirement to take exams?')

NameError: name 'top_k_search' is not defined

## 📋 Step 11: Retrieval Test Suite & Evaluation Report

In [ ]:
TEST_QUERIES = [
    ('attendance',    'What is the minimum attendance requirement to appear for exams?'),
    ('grading',       'Explain the grading system and grade points for 90+ marks.'),
    ('scholarship',   'Who is eligible for a full tuition fee waiver merit scholarship?'),
    ('library',       'How many books can a student borrow and for how many days?'),
    ('hostel_timing', 'What are the hostel gate closing timings on weekdays and weekends?'),
    ('cse_credits',   'How many total credits are required to complete the B.Tech CSE program?'),
    ('internal_marks','What percentage do internal assessments contribute to the final grade?'),
    ('electives',     'From which semester are electives offered in the CSE department?'),
]

results_log = []

print('='*70)
print('  RUNNING RETRIEVAL EVALUATION TEST SUITE')
print('='*70)

for label, query in TEST_QUERIES:
    raw = top_k_search(vectorstore, query, k=DEFAULT_K)
    filtered = filter_by_score(raw, DEFAULT_THRESHOLD)
    top_score = raw[0][1] if raw else 0.0
    passed = len(filtered) > 0 and top_score >= DEFAULT_THRESHOLD

    results_log.append({
        'label': label,
        'query': query,
        'top_score': top_score,
        'kept_chunks': len(filtered),
        'passed': passed
    })

print('\n' + '═'*70)
print(f"  📊 TEST REPORT: {sum(r['passed'] for r in results_log)}/{len(TEST_QUERIES)} TESTS PASSED")
print('═'*70)
for r in results_log:
    icon = '✅ PASS' if r['passed'] else '❌ FAIL'
    print(f"  {icon} | [{r['label']:<15}] Score: {r['top_score']:.3f} | Chunks: {r['kept_chunks']}")
print('═'*70)

## 💬 Step 12: Interactive Chat Interface

In [ ]:
print('🎓 Welcome to the College Academic Assistant!')
print('Type your question below and press Enter. (Type "exit" or "quit" to stop)\n')

while True:
    try:
        user_q = input('Ask a question: ').strip()
        if not user_q:
            continue
        if user_q.lower() in ['exit', 'quit', 'q']:
            print('👋 Exiting assistant. Good luck with your studies!')
            break
        ask(user_q, show_sources=True)
    except KeyboardInterrupt:
        print('\nChat terminated.')
        break

In [26]:
!pip install -q langgraph

In [27]:
from typing import TypedDict, List, Dict

class AgentState(TypedDict, total=False):
    user_input: str
    route: str
    answer: str

    subjects: List[str]
    exam_dates: Dict[str, str]
    study_hours: float
    study_plan: str

In [28]:
def router(state):
    query = state["user_input"].lower()

    study_keywords = [
        "study plan",
        "study timetable",
        "study schedule",
        "timetable",
        "schedule",
        "exam preparation",
        "prepare for exam"
    ]

    if any(keyword in query for keyword in study_keywords):
        state["route"] = "study_planner"
    else:
        state["route"] = "rag"

    return state

In [30]:
def rag_node(state):
    question = state["user_input"]
    answer = ask(question)
    state["answer"] = answer
    return state

In [36]:
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

print("API key loaded:", GOOGLE_API_KEY is not None)

API key loaded: True


In [41]:
from langchain_google_genai import ChatGoogleGenerativeAI

study_llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.2
)

In [42]:
def study_planner_node(state):
    subjects = state.get("subjects", [])
    exam_dates = state.get("exam_dates", {})
    study_hours = state.get("study_hours", 0)

    prompt = f"""
You are a college study planning assistant.

Create a realistic study timetable.

Subjects:
{subjects}

Exam dates:
{exam_dates}

Available study hours per day:
{study_hours}

Requirements:
- Prioritize exams that are closer.
- Give appropriate time to each subject.
- Include revision.
- Include reasonable breaks.
- Do not exceed the available study hours.

Return the timetable in a clear format.
"""

    response = study_llm.invoke(prompt)

    state["study_plan"] = response.content
    state["answer"] = response.content

    return state

In [39]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(AgentState)

builder.add_node("router", router)
builder.add_node("rag", rag_node)
builder.add_node("study_planner", study_planner_node)

builder.add_edge(START, "router")

builder.add_conditional_edges(
    "router",
    lambda state: state["route"],
    {
        "rag": "rag",
        "study_planner": "study_planner"
    }
)

builder.add_edge("rag", END)
builder.add_edge("study_planner", END)

graph = builder.compile()

print("Graph compiled successfully!")

Graph compiled successfully!


In [43]:
result = graph.invoke({
    "user_input": "Create a study plan for my exams",
    "subjects": ["DBMS", "Computer Networks", "AI"],
    "exam_dates": {
        "DBMS": "2026-10-10",
        "Computer Networks": "2026-10-13",
        "AI": "2026-10-16"
    },
    "study_hours": 4
})

print(result["answer"])

[{'type': 'text', 'text': 'Here is a realistic, structured study plan tailored to your exam schedule, prioritizing the earliest exams while keeping future subjects fresh.\n\n---\n\n### **1. Daily Block Schedule (4 Study Hours + Breaks)**\n\nTo avoid fatigue and maximize retention, each 4-hour daily allotment is divided using the **Pomodoro-style block method**:\n\n| Time Block | Duration | Activity |\n| :--- | :--- | :--- |\n| **Block 1** | 90 mins (1.5 hrs) | Deep Study / Heavy Topics |\n| *Break* | *15 mins* | *Rest, hydrate, step away from screens* |\n| **Block 2** | 90 mins (1.5 hrs) | Problem Solving / Concept Building |\n| *Break* | *15 mins* | *Light walk / snack* |\n| **Block 3** | 60 mins (1.0 hr) | Active Recall, Practice Questions & Revision |\n| **Total** | **4.0 Hours Study** | **30 Mins Total Breaks** |\n\n---\n\n### **2. Strategic Phase-by-Phase Plan**\n\n* **Phase 1 (Oct 2 – Oct 5):** Multi-subject coverage, skewed toward DBMS.\n* **Phase 2 (Oct 6 – Oct 9):** Final spri

In [44]:
result = graph.invoke({
    "user_input": "What are the units in Object Oriented Programming?"
})

print(result["answer"])

[Mockup Answer] Based on the context of your question 'What are the units in Object Oriented Programming?', Object Oriented Programming consists of classes, objects, inheritance, polymorphism, encapsulation, and abstraction.
